# Lab 11: MNIST and Convolutional Neural Network (CNN)
### TF2 / Keras 스타일 — Conv2D + MaxPooling + Dense

In [ ]:
# Lab 11 MNIST and Convolutional Neural Network
import numpy as np
import tensorflow as tf
import random
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

In [ ]:
# 데이터 로드 및 전처리
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = x_train / 255.0
x_test  = x_test  / 255.0

# CNN 입력 shape: (samples, height, width, channels)
x_train = x_train.reshape(x_train.shape[0], 28, 28, 1)
x_test  = x_test.reshape(x_test.shape[0],   28, 28, 1)

# one-hot encoding
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test  = tf.keras.utils.to_categorical(y_test,  10)

print("x_train:", x_train.shape, "y_train:", y_train.shape)
print("x_test: ", x_test.shape,  "y_test: ", y_test.shape)

In [ ]:
# hyper parameters
learning_rate   = 0.001
training_epochs = 12
batch_size      = 128

In [ ]:
# 모델 구성
tf.model = tf.keras.Sequential()

# L1: Conv + MaxPool  (28x28x1 → 13x13x16)
tf.model.add(tf.keras.layers.Conv2D(filters=16, kernel_size=(3,3),
                                    input_shape=(28, 28, 1), activation='relu'))
tf.model.add(tf.keras.layers.MaxPooling2D(pool_size=(2,2)))

# L2: Conv + MaxPool  (13x13x16 → 5x5x32)
tf.model.add(tf.keras.layers.Conv2D(filters=32, kernel_size=(3,3), activation='relu'))
tf.model.add(tf.keras.layers.MaxPooling2D(pool_size=(2,2)))

# L3: Fully Connected
tf.model.add(tf.keras.layers.Flatten())
tf.model.add(tf.keras.layers.Dense(units=10,
                                   kernel_initializer='glorot_normal',
                                   activation='softmax'))

tf.model.compile(loss='categorical_crossentropy',
                 optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                 metrics=['accuracy'])
tf.model.summary()

In [ ]:
# 학습
history = tf.model.fit(x_train, y_train,
                       batch_size=batch_size,
                       epochs=training_epochs,
                       validation_split=0.1)

In [ ]:
# 학습 곡선 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['loss'],     'b-o', label='Train Loss',     markersize=4)
ax1.plot(history.history['val_loss'], 'r-o', label='Val Loss',       markersize=4)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history.history['accuracy'],     'b-o', label='Train Accuracy', markersize=4)
ax2.plot(history.history['val_accuracy'], 'r-o', label='Val Accuracy',   markersize=4)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Training & Validation Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 평가
evaluation = tf.model.evaluate(x_test, y_test, verbose=0)
print(f'Test Loss:     {evaluation[0]:.4f}')
print(f'Test Accuracy: {evaluation[1]*100:.2f}%')

In [ ]:
# 랜덤 예측 이미지 9장 시각화
y_predicted = tf.model.predict(x_test, verbose=0)

fig, axes = plt.subplots(3, 3, figsize=(7, 7))
fig.suptitle(f'Random Predictions  (Test Accuracy: {evaluation[1]*100:.2f}%)', fontsize=12)

for ax in axes.flatten():
    r      = random.randint(0, x_test.shape[0]-1)
    img    = x_test[r].reshape(28, 28)
    label  = np.argmax(y_test[r])
    pred   = np.argmax(y_predicted[r])
    color  = 'green' if pred == label else 'red'
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Label:{label}  Pred:{pred}', fontsize=9, color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()